# FINPLE Step 108-12B — US Dividend Overlay Chunked Colab

This notebook uses the safer chunked script:

```text
build_us_dividend_overlay_chunked.py
finple_app_candidates_6000_balanced_v1.csv
```

The old full-run output names such as `us_dividend_overlay_test.csv` are no longer used.


In [ ]:
# 1. Install dependencies
!pip -q install yfinance pandas


## 2. Upload required files

Upload exactly these two files:

```text
build_us_dividend_overlay_chunked.py
finple_app_candidates_6000_balanced_v1.csv
```


In [ ]:
from google.colab import files
uploaded = files.upload()


## 3. First test run — 20 rows

Use this to confirm that the chunked script works and creates output files.


In [ ]:
!python build_us_dividend_overlay_chunked.py \
  --input finple_app_candidates_6000_balanced_v1.csv \
  --out-runtime us_dividend_overlay_20260527_part0000_0020.csv \
  --out-audit us_dividend_overlay_20260527_part0000_0020_audit.csv \
  --out-summary us_dividend_overlay_20260527_part0000_0020_summary.json \
  --as-of 2026-05-27 \
  --start 0 \
  --limit 20 \
  --checkpoint-every 5


## 4. Inspect 20-row test output

This cell must read the same filenames created in step 3.


In [ ]:
import pandas as pd, json, os
runtime_file = 'us_dividend_overlay_20260527_part0000_0020.csv'
summary_file = 'us_dividend_overlay_20260527_part0000_0020_summary.json'
print('runtime exists:', os.path.exists(runtime_file), runtime_file)
print('summary exists:', os.path.exists(summary_file), summary_file)
test_df = pd.read_csv(runtime_file)
display(test_df.head(30))
with open(summary_file, 'r', encoding='utf-8') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))


## 5. Production chunk run — 100 rows

After the 20-row test succeeds, run chunks of 100. Change `--start` and file names for each chunk.


In [ ]:
!python build_us_dividend_overlay_chunked.py \
  --input finple_app_candidates_6000_balanced_v1.csv \
  --out-runtime us_dividend_overlay_20260527_part0000_0100.csv \
  --out-audit us_dividend_overlay_20260527_part0000_0100_audit.csv \
  --out-summary us_dividend_overlay_20260527_part0000_0100_summary.json \
  --as-of 2026-05-27 \
  --start 0 \
  --limit 100 \
  --checkpoint-every 25


## 6. Inspect 100-row chunk


In [ ]:
chunk_df = pd.read_csv('us_dividend_overlay_20260527_part0000_0100.csv')
sample = ['SPY','VOO','QQQ','SCHD','JEPI','JEPQ','TLT','BND','VNQ','AAPL','MSFT','KO','JNJ','JPM','XOM','AMZN','TSLA','SNOW']
display(chunk_df[chunk_df['ticker'].isin(sample)].sort_values('ticker'))
display(chunk_df.head(20))
with open('us_dividend_overlay_20260527_part0000_0100_summary.json', 'r', encoding='utf-8') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))


## 7. Combine downloaded/created chunks

Run this after multiple `part*.csv` files are created.


In [ ]:
import glob
files = sorted(glob.glob('us_dividend_overlay_20260527_part*.csv'))
print(files)
combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
combined = combined.drop_duplicates(['market', 'ticker'], keep='last')
combined.to_csv('us_dividend_overlay_20260527.csv', index=False, encoding='utf-8-sig')
print(combined.shape)
display(combined.head())


## 8. Download outputs


In [ ]:
files.download('us_dividend_overlay_20260527.csv')
